In [0]:
"""
===============================================================
Module: Superstore Master Run ID Orchestrator Initialization

Purpose:
    This module initializes the Superstore ETL pipeline by generating a unique Master Run ID,
    setting up runtime environment variables, and logging pipeline execution details. It serves as the 
    entry point for orchestrating the Superstore Medallion Lakehouse pipeline, ensuring proper execution 
    context and observability across the pipeline stages.

Key Features:
1. Master Run ID Generation:
    - Creates a unique `master_run_id` for each pipeline execution, enabling traceability and structured logging.
    - The `master_run_id` is used to track the entire pipeline run, including all child orchestrators.

2. Runtime Environment Setup:
    - Reads job parameters such as `SUPERSTORE_ENV`, `SUPERSTORE_PIPELINE_NAME`, and `SUPERSTORE_PIPELINE_VERSION` from Databricks job widgets.
    - Sets these parameters as environment variables for downstream tasks in the pipeline.

3. Logging & Metrics:
    - Uses structured logging to track key events, such as the creation of the `master_run_id` and the start of the pipeline.
    - Logs messages are tagged with the `master_run_id` for consistency across the pipeline.

4. Error Handling:
    - If the `master_run_id` creation fails, an error is logged, and the pipeline exits gracefully, preventing further execution.

5. Task Sharing:
    - Shares the `master_run_id` and environment variables with downstream DAG tasks using Databricks job task values, ensuring that all tasks have consistent execution context.

6. Production-Ready:
    - The module is designed to handle large-scale, production-level ETL pipelines using Delta Lake's Medallion Architecture.
    - It ensures proper execution context for logging, metrics, and observability across multiple layers of the pipeline.

Imports:
    - `sys`: Used to handle system-level operations and error exits.
    - `superstore_logger`: Custom logging module to handle structured logs with metadata.
    - `os`: Used to set runtime environment variables.
    - `dbutils`: Databricks utility module to manage widgets and task values.

Usage:
    This module is executed as part of the ETL pipeline setup, initializing the pipeline environment and logging key events.

    Example usage:
        import superstore_master_orchestrator
        superstore_master_orchestrator.initialize_pipeline()

===============================================================
"""

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

import os, sys
from superstore_logger import get_superstore_logger, log_event, create_master_run_id

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

# -----------------------------
# Read job parameters (from DAB)
# -----------------------------
dbutils.widgets.text("SUPERSTORE_ENV", "dev")
dbutils.widgets.text("SUPERSTORE_PIPELINE_NAME", "superstore_etl_pipeline")
dbutils.widgets.text("SUPERSTORE_PIPELINE_VERSION", "1.0")

# Run-mode parameters. Declared here so they are visible when the notebook is
# opened directly; in a job run the job parameter of the same key wins.
#
# The Jobs UI has no enum type for job parameters - "Run now with different
# parameters" is free text - so the valid values cannot be offered as a
# dropdown there. They are listed in the job description, and this task
# validates them below so a typo fails in seconds instead of after Bronze has
# started.
dbutils.widgets.text("run_mode", "incremental")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("dry_run", "false")

env = dbutils.widgets.get("SUPERSTORE_ENV")
pipeline_name = dbutils.widgets.get("SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.widgets.get("SUPERSTORE_PIPELINE_VERSION")

# -----------------------------
# Set runtime environment variables
# -----------------------------
os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Initialize logger
# -----------------------------
logger_master = get_superstore_logger("superstore_master_orchestrator")

try:
    # -----------------------------
    # Create Master Run ID
    # -----------------------------
    master_run_id: str = create_master_run_id()  # Explicit type annotation
    log_event(
        logger_master,
        "INFO",
        "Master Run ID for superstore medallion lakehouse pipeline successfully created",
        master_run_id=master_run_id,
        layer=None
    )

except Exception as e:
    # -----------------------------
    # Handle failure in master ID creation
    # -----------------------------
    master_run_id: str = "MASTER_ID_CREATION_FAILED"
    log_event(
        logger_master,
        "ERROR",
        f"Failed to create Master Run ID: {str(e)}",
        master_run_id=master_run_id,
        layer=None
    )
    # Optionally exit immediately if this is critical
    import sys
    sys.exit(f"Critical: Could not generate master_run_id. Error: {e}")

# -----------------------------
# Log pipeline start
# -----------------------------
log_event(
    logger_master,
    "INFO",
    "Initiating Superstore medallion lakehouse Pipeline using Delta Lake Medallion Architecture",
    master_run_id=master_run_id,
    layer=None
)


# -----------------------------
# Share master_run_id with DAG tasks
# -----------------------------
dbutils.jobs.taskValues.set(key="master_run_id", value=master_run_id)

dbutils.jobs.taskValues.set(key="SUPERSTORE_ENV", value=env)
dbutils.jobs.taskValues.set(key="SUPERSTORE_PIPELINE_NAME", value=pipeline_name)
dbutils.jobs.taskValues.set(key="SUPERSTORE_PIPELINE_VERSION", value=pipeline_version)

print(f"SUPERSTORE DATA PLATFORMS")
print(f"ENV: {env}")
print(f"PIPELINE_NAME: {pipeline_name}")
print(f"PIPELINE_VERSION: {pipeline_version}")
print(f"MASTER_RUN_ID: {master_run_id}")


# -----------------------------
# Validate run parameters and print the run plan
# -----------------------------
# This task is the entry point of the DAG and writes no data, so it is the
# cheapest place to reject a bad parameter: an unknown run_mode or a missing
# start_date fails here in seconds rather than after serverless compute has
# started for Bronze.
#
# allow_full_refresh=True is deliberate. This is validation, not enforcement -
# the full_refresh gate belongs in the four orchestrators that actually write,
# and is left there. Here we only check that the parameters make sense.
from superstore_backfill_utils import get_backfill_config, VALID_RUN_MODES

try:
    run_config = get_backfill_config(dbutils, allow_full_refresh=True)
except ValueError as e:
    log_event(
        logger_master,
        "ERROR",
        f"Invalid run parameters: {e}",
        master_run_id=master_run_id,
        layer=None
    )
    raise

run_mode = run_config["mode"]
dry_run = run_config["dry_run"]

# What this run will actually do, spelled out. An operator who does not know
# the modes should be able to read this and stop the run if it is not what
# they intended.
_window = "all data (no window)"
if run_config["is_windowed"]:
    _window = (
        f"{run_config['start_date'].strftime('%Y-%m-%d')}"
        f" to {run_config['end_date'].strftime('%Y-%m-%d')} (by INGESTION date, not order date)"
    )

_bronze = "RUNS - re-reads from source" if run_config["reads_source"] else "SKIPPED - re-derives from existing Bronze"
if run_mode == "incremental":
    _bronze = "RUNS - new files only"

print("")
print("RUN PLAN")
print(f"  run_mode   : {run_mode}   (valid: {', '.join(VALID_RUN_MODES)})")
print(f"  window     : {_window}")
print(f"  bronze     : {_bronze}")
print(f"  writes     : {'NO - dry run, every task reports impact and exits' if dry_run else 'YES'}")

log_event(
    logger_master,
    "INFO",
    f"Run plan: mode={run_mode} dry_run={dry_run} window={_window}",
    run_mode=run_mode,
    dry_run=dry_run,
    reads_source=run_config["reads_source"],
    master_run_id=master_run_id,
    layer=None
)

# Share the resolved mode with downstream tasks for logging and metrics.
dbutils.jobs.taskValues.set(key="run_mode", value=run_mode)